# Computación Adiabática y QUBO

**Objetivo:** formular MAX-CUT como problema QUBO y resolverlo por simulated annealing.

El principio adiabático: si el Hamiltoniano evoluciona suficientemente despacio desde H_i (fácil) hasta H_f (problema), el sistema permanece en su ground state.
QUBO: E(x) = xᵀQx,  x ∈ {0,1}^n  — el ground state da la solución óptima.

In [ ]:
import numpy as np
from itertools import product

# QUBO para MAX-CUT: grafo de 4 nodos
# Para arista (i,j): Q_ij = Q_ji = -1, Q_ii += 1, Q_jj += 1 (dividido)
V = 4
E = [(0,1),(1,2),(2,3),(0,3),(0,2)]  # grafo con diagonal

Q = np.zeros((V, V))
for i, j in E:
    Q[i,i] += 1
    Q[j,j] += 1
    Q[i,j] -= 1
    Q[j,i] -= 1
Q /= 2  # normalizar para que la energía = núm. aristas cortadas

print('Matriz Q:')
print(Q.round(2))

In [ ]:
# Evaluación de energía QUBO
def qubo_energy(x, Q):
    return x @ Q @ x

# Búsqueda por fuerza bruta
best_energy = np.inf
best_x = None
results = []
for bits in product([0,1], repeat=V):
    x = np.array(bits, dtype=float)
    e = qubo_energy(x, Q)
    results.append((e, bits))
    if e < best_energy:
        best_energy = e
        best_x = bits

results.sort()
print('Top 5 soluciones QUBO:')
for e, bits in results[:5]:
    print(f'  x={bits}: E={e:.2f}  (corte={int(-e*2+len(E))} aristas... aprox)')
print(f'\nMejor solución: {best_x}, E={best_energy:.2f}')

In [ ]:
# MAX-CUT real: cuántas aristas corta cada solución
def max_cut_value(x, edges):
    return sum(1 for i,j in edges if x[i] != x[j])

best_cut = 0
best_partition = None
for bits in product([0,1], repeat=V):
    cut = max_cut_value(bits, E)
    if cut > best_cut:
        best_cut = cut
        best_partition = bits

print(f'MAX-CUT real: {best_cut} aristas')
print(f'Partición óptima: {best_partition}')

## Simulated Annealing

Simulated Annealing imita el enfriamiento lento de metales para encontrar el mínimo global de QUBO.
Acepta soluciones peores con probabilidad e^(-ΔE/T) para escapar de mínimos locales.

In [ ]:
def simulated_annealing(Q, n_steps=5000, T_start=1.0, T_end=0.01):
    n = Q.shape[0]
    x = np.random.randint(0, 2, n).astype(float)
    E = qubo_energy(x, Q)
    best_x, best_E = x.copy(), E
    
    for step in range(n_steps):
        T = T_start * (T_end/T_start) ** (step/n_steps)  # enfriamiento exponencial
        # Flip bit aleatorio
        i = np.random.randint(n)
        x_new = x.copy(); x_new[i] = 1 - x_new[i]
        E_new = qubo_energy(x_new, Q)
        dE = E_new - E
        if dE < 0 or np.random.rand() < np.exp(-dE / T):
            x, E = x_new, E_new
        if E < best_E:
            best_E, best_x = E, x.copy()
    return best_x, best_E

np.random.seed(42)
x_sa, E_sa = simulated_annealing(Q)
cut_sa = max_cut_value(x_sa.astype(int), E)
print(f'Simulated Annealing: x={x_sa.astype(int)}, E={E_sa:.2f}, corte={cut_sa}')
print(f'Óptimo fuerza bruta:  x={list(best_partition)}, corte={best_cut}')

In [ ]:
# Energía de Ising: transformar QUBO → Ising (x_i = (1-s_i)/2)
def qubo_to_ising(Q):
    n = Q.shape[0]
    J = np.zeros((n,n))
    h = np.zeros(n)
    for i in range(n):
        for j in range(n):
            J[i,j] = Q[i,j]/4
            h[i] -= Q[i,j]/2
    return J, h

J, h = qubo_to_ising(Q)
print('Hamiltoniano de Ising H = -Σ J_ij s_i s_j - Σ h_i s_i')
print('J (off-diagonal):')
print((J - np.diag(np.diag(J))).round(3))
print('h:', h.round(3))

## Conexión con QAOA

El Hamiltoniano de costo de QAOA es exactamente el mismo QUBO en la base computacional.
La diferencia: QAOA lo resuelve con circuitos cuánticos, SA lo resuelve clásicamente.

In [ ]:
print('Tabla comparativa:')
print('─'*60)
print(f'{'Método':<20} {'Calidad':<15} {'Escalado':<15} {'Hardware'}')
print('─'*60)
print(f'{'Fuerza bruta':<20} {'Exacto':<15} {'O(2^n)':<15} {'CPU clásica'}')
print(f'{'Simul. Annealing':<20} {'Heurístico':<15} {'O(n²·steps)':<15} {'CPU/GPU'}')
print(f'{'QAOA p=1':<20} {'Aprox.':<15} {'O(n·p)':<15} {'Cuántico (ruidoso)'}')
print(f'{'D-Wave Annealing':<20} {'Heurístico':<15} {'O(µs)':<15} {'QPU D-Wave'}')
print('─'*60)

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/53_qubo_dwave.ipynb`
- **QAOA:** `09_qaoa_paso_a_paso.ipynb`
- **Siguiente guía:** `15_fault_tolerant.ipynb`